# Splitting dataset into Train, Val, Test

In [ ]:
"""
Code to split a two-class image dataset into train / val / test sets (70:15:15).

Input structure:
    dataset/
        overthinking/
        nooverthinking/

Output structure:
    split_dataset/
        train/
            overthinking/
            nooverthinking/
        val/
            overthinking/
            nooverthinking/
        test/
            overthinking/
            nooverthinking/
"""

import shutil
import random
from pathlib import Path

# ── Configure your folders here ───────────────────────────────────────────────
INPUT_FOLDER  = "D:/Excessive_Thought/Expert2/Paper4_images_E2"         # <-- folder containing overthinking & nooverthinking
OUTPUT_FOLDER = "D:/Excessive_Thought/Expert2/Overthining_Before_aug"   # <-- where train/val/test will be created
# ─────────────────────────────────────────────────────────────────────────────

CLASSES    = ["overthinking", "nooverthinking"]
SPLITS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SUPPORTED  = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif", ".tiff"}
RANDOM_SEED = 42


def split_dataset(input_folder: str, output_folder: str) -> None:
    input_path  = Path(input_folder)
    output_path = Path(output_folder)

    # Validate input
    for cls in CLASSES:
        cls_path = input_path / cls
        if not cls_path.exists():
            raise FileNotFoundError(f"Class folder not found: {cls_path}")

    # Create all output directories upfront
    for split in SPLITS:
        for cls in CLASSES:
            (output_path / split / cls).mkdir(parents=True, exist_ok=True)

    random.seed(RANDOM_SEED)

    overall = {"train": 0, "val": 0, "test": 0}

    for cls in CLASSES:
        cls_path = input_path / cls
        images   = sorted([
            f for f in cls_path.iterdir()
            if f.is_file() and f.suffix.lower() in SUPPORTED
        ])

        if not images:
            print(f"    No images found in '{cls_path}', skipping.")
            continue

        random.shuffle(images)
        total = len(images)

        n_train = int(total * SPLITS["train"])
        n_val   = int(total * SPLITS["val"])
        # test gets the remainder to avoid rounding loss
        n_test  = total - n_train - n_val

        split_data = {
            "train": images[:n_train],
            "val"  : images[n_train : n_train + n_val],
            "test" : images[n_train + n_val :],
        }

        print(f"\nClass: {cls}  (total: {total})")
        print(f"  train: {n_train}  |  val: {n_val}  |  test: {n_test}")

        for split, files in split_data.items():
            dest_dir = output_path / split / cls
            for f in files:
                shutil.copy2(f, dest_dir / f.name)
            overall[split] += len(files)

    total_all = sum(overall.values())
    print(f"\n{'─'*45}")
    print(f"{'Split':<10} {'Images':>8}  {'%':>6}")
    print(f"{'─'*45}")
    for split, count in overall.items():
        pct = count / total_all * 100 if total_all else 0
        print(f"{split:<10} {count:>8}  {pct:>5.1f}%")
    print(f"{'─'*45}")
    print(f"{'Total':<10} {total_all:>8}  100.0%")
    print(f"\nOutput saved to: {output_path.resolve()}")


split_dataset(INPUT_FOLDER, OUTPUT_FOLDER)

# Image Augmentation

In [ ]:
"""
Code to apply augmentation techniques to images in the train folder only.
Original images are retained. Augmented images are saved in the same folder.

Techniques applied:
  - Horizontal Flip
  - Vertical Flip
  - Rotation
  - Shear

Input/Output structure:
    split_dataset/
        train/
            overthinking/     
            nooverthinking/   
        val/   (untouched)
        test/  (untouched)
"""

from pathlib import Path
from PIL import Image
import numpy as np
from skimage.transform import AffineTransform, warp

# ── Configure your folder here ────────────────────────────────────────────────
TRAIN_FOLDER = "D:/Excessive_Thought/Expert2/Overthinking_E2/train"  # <-- path to your train folder
# ─────────────────────────────────────────────────────────────────────────────

CLASSES   = ["overthinking", "nooverthinking"]
SUPPORTED = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

# Rotation angles to generate (one augmented image per angle)
ROTATION_ANGLES = [10]

# Shear range (single mild shear)
SHEAR_VALUES = [0.1]


def save_augmented(img: Image.Image, src_path: Path, suffix: str) -> None:
    """Save augmented image with a descriptive suffix appended to filename."""
    out_name = f"{src_path.stem}_{suffix}{src_path.suffix}"
    out_path = src_path.parent / out_name
    img.save(out_path)
    print(f"      saved → {out_name}")


def apply_shear(img: Image.Image, shear: float) -> Image.Image:
    """Apply shear transform using skimage and return a PIL Image."""
    img_array = np.array(img)
    tf = AffineTransform(shear=shear)
    # warp returns float64 in [0,1] — rescale back to uint8
    sheared = warp(img_array, inverse_map=tf.inverse,
                   mode="edge", preserve_range=True)
    return Image.fromarray(sheared.astype(np.uint8))


def augment_image(img_path: Path) -> None:
    with Image.open(img_path) as img:
        # Ensure consistent mode
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGB")
        img = img.copy()  # detach from file handle

    print(f"  ▸ {img_path.name}")

    # 1. Horizontal Flip
    h_flip = img.transpose(Image.FLIP_LEFT_RIGHT)
    save_augmented(h_flip, img_path, "hflip")

    # 2. Vertical Flip
    v_flip = img.transpose(Image.FLIP_TOP_BOTTOM)
    save_augmented(v_flip, img_path, "vflip")

    # 3. Rotation 
    for angle in ROTATION_ANGLES:
        rotated = img.rotate(angle, expand=True)
        save_augmented(rotated, img_path, f"rot{angle}")

    # 4. Shear
    for shear_val in SHEAR_VALUES:
        sheared = apply_shear(img, shear_val)
        label = f"shear_pos" if shear_val > 0 else "shear_neg"
        save_augmented(sheared, img_path, label)


def augment_train(train_folder: str) -> None:
    train_path = Path(train_folder)

    if not train_path.exists():
        raise FileNotFoundError(f"Train folder not found: {train_path}")

    total_original  = 0
    total_augmented = 0

    for cls in CLASSES:
        cls_path = train_path / cls
        if not cls_path.exists():
            print(f" Class folder not found, skipping: {cls_path}")
            continue

        # Collect only original images (exclude already-augmented ones)
        images = sorted([
            f for f in cls_path.iterdir()
            if f.is_file() and f.suffix.lower() in SUPPORTED
            and not any(tag in f.stem for tag in
                        ["hflip", "vflip", "rot10", "shear"])
        ])

        print(f"\n── Class: {cls}  ({len(images)} original images) ──")

        per_image_aug = len(ROTATION_ANGLES) + len(SHEAR_VALUES) + 2  # flips
        for img_path in images:
            augment_image(img_path)

        total_original  += len(images)
        total_augmented += len(images) * per_image_aug

    print(f"\n{'─'*50}")
    print(f"  Original images  : {total_original}")
    print(f"  Augmented images : {total_augmented}")
    print(f"  Total in train   : {total_original + total_augmented}")
    print(f"{'─'*50}")
    print(f"\nAugmentation complete. Files saved in: {train_path.resolve()}")


augment_train(TRAIN_FOLDER)